# Exploration of JSON-LD Contexts as LLM Context for Agentic Retreivers
Large Language Models (LLMs) have revolutionized artificial intelligence (AI) by enabling humanlike text generation and natural language understanding. However, their reliance on static training
data limits their ability to respond to dynamic, real-time queries, resulting in outdated or inaccurate outputs. Retrieval-Augmented Generation (RAG) has emerged as a solution, enhancing LLMs by integrating real-time data retrieval to provide contextually relevant and up-to-date responses. Despite its promise, traditional RAG systems are constrained by static workflows and lack the adaptability required for multi-step reasoning and complex task management. Agentic Retrieval-Augmented Generation (Agentic RAG) transcends these limitations by embedding autonomous AI agents into the RAG pipeline. These agents leverage agentic design patterns—reflection, planning, tool use, and multi-agent collaboration—to dynamically manage retrieval strategies, iteratively refine contextual understanding, and adapt workflows to meet complex task requirements. This integration enables Agentic RAG systems to deliver unparalleled flexibility, scalability, and context-awareness across diverse applications.

JSON-LD can provide queue tokens to agentic retrievers to discover more information and include that information in the LLM context. This Solve-It is a exploration of that approach using the Solve-It methodology. The hypothysis is that Ontology Design Patterns, and Modular Ontologies within the JSON-LD context give more retrival context to the llm. 

## Plan
- Use GitHub to store "JSON-LD Ontology Contexts"
- Use Claudette to prototype agentic system in Solve-It
- LLM retreives Whole Ontology JSON-LD from "GitHub" and uses the Ontology definition information to provide context.
- LLM uses OPLaX vocabulary to understand modular structure, patterns and annotations.
- Each ontology entity must be LLM friendly
- Ontology hierarchy structure for each module should be LLM friendly.
- The Main Ontology JSON-LD contains the URIs for dereferencing each module and a description such that the Agentic workflow can decide when it needs more context.

In [ ]:
from fastcore.utils import *
import fastcore.all as fc, re, math, itertools, functools, numpy as np, types, typing, dataclasses, matplotlib.pyplot as plt, collections, regex
from regex import search
from collections import Counter
from collections.abc import Iterable
from SPARQLWrapper import SPARQLWrapper
from rdflib import Graph
from claudette import *
import httpx
np.set_printoptions(linewidth=150, suppress=True)
plt.rcParams['figure.dpi'] = 50

**AI Prompt**

Hi solveit! I am Chuck, and I am working with you today! I'm an experienced Python coder, experienced with semantic web technologies like OWL,RDF and SPARQL, and I enjoy learning new things. When writing code for me, please follow fast.ai coding style: avoid comments; place single line if/loop/etc constructs on the same line; use concise variable names where appropriate. Follow the style you will see in my messages, as we work on this dialogue together. Are you ready?

**AI Response**

Hi Chuck! Absolutely, I'm ready to work with you! It's great to meet someone experienced with semantic web technologies - that will be particularly relevant given your interesting exploration of JSON-LD contexts for agentic retrievers.

I see you've already set up some useful imports and are planning to explore how JSON-LD ontology contexts can enhance LLM retrieval capabilities. I'll follow the fast.ai coding style you've described and match your approach.

Would you like to start by setting up a basic JSON-LD context structure, or would you prefer to begin with a different aspect of the exploration? I'm particularly interested in how you envision using OPLaX vocabulary for the modular structure understanding.

## The agentic patterns
### Reflection Pattern
This pattern allows the LLM to reflect and critique its outputs, following the next steps:

- The LLM generates a candidate output. 
- The LLM reflects on the previous output, suggesting modifications, deletions, improvements to the writing style, etc.
- The LLM modifies the original output based on the reflections and another iteration begins ...

### Tool Pattern
This pattern allows the LLMs to use Tools functions defined by Claudette as part of the agentic workflow

### Planning
That is exactly what the Planning Pattern provides; ways for the LLM to break a task into smaller, more easily accomplished subgoals without losing track of the end goal.

The most paradigmatic example of the planning pattern is the ReAct technique.

**AI Prompt**

I think we want to build a very simple toy model of an agentic workflow. The workflow should talk a sample JSON-LD that has a @context and a @graph along with some tools that help the agentic workflow understand the JSON-LD context. The tools should be able to parse the JSON-LD context and return the relevant information to the LLM. The vocabulary in the @context should be a github raw link to the JSON-LD vocabulary file.

**AI Response**

Great idea! Let's build this step by step. First, let's create some basic tools for handling JSON-LD contexts and a simple test vocabulary. 

Let's start with a tool that can fetch and parse JSON-LD from a URL, and another that can extract context information. Here's what I suggest:

```python
import json
from pathlib import Path

def fetch_jsonld(
    url: str  # URL of JSON-LD file
) -> dict:   # Parsed JSON-LD content
    "Fetches JSON-LD from a URL"
    return httpx.get(url).json()

def extract_context(
    jsonld: dict  # JSON-LD document
) -> dict:       # Context information
    "Extracts and processes @context from JSON-LD"
    ctx = jsonld.get('@context', {})
    return ctx if isinstance(ctx, dict) else fetch_jsonld(ctx) if isinstance(ctx, str) else {}
```

Now let's create a simple test vocabulary and put it in a GitHub gist. Would you like me to create one, or do you have a specific vocabulary structure in mind that you'd like to use for testing? 

We can then create a Chat instance with these tools and test the agentic workflow with a sample JSON-LD document. How would you like to structure the test vocabulary?

In [ ]:
import json
from pathlib import Path

def fetch_jsonld(
    url: str  # URL of JSON-LD file
) -> dict:   # Parsed JSON-LD content
    "Fetches JSON-LD from a URL"
    return httpx.get(url).json()

def extract_context(
    jsonld: dict  # JSON-LD document
) -> dict:       # Context information
    "Extracts and processes @context from JSON-LD"
    ctx = jsonld.get('@context', {})
    return ctx if isinstance(ctx, dict) else fetch_jsonld(ctx) if isinstance(ctx, str) else {}

jsonld = fetch_jsonld("https://raw.githubusercontent.com/LA3D/Agentic-JSONLD-Experiment/refs/heads/main/llm-context/sample.jsonld")
print(jsonld)
context = extract_context(jsonld)
context

{'@context': {'owl': 'http://www.w3.org/2002/07/owl#', 'dct': 'http://purl.org/dc/terms/', 'vann': 'http://purl.org/vocab/vann/', 'foaf': 'http://xmlns.com/foaf/0.1/', 'prov': 'http://www.w3.org/ns/prov#', 'oplax': 'https://example.com/oplax#'}, '@graph': [{'@id': 'https://example.com/supply-chain-document-workflow#', '@type': 'owl:Ontology', 'dct:title': 'Supply Chain Document Workflow Ontology', 'dct:description': 'Provides a structured set of modules (Core, Ordering, Billing, Logistics) to describe documents and processes within a supply chain workflow.', 'dct:publisher': {'@id': 'https://example.com/organization/PublisherOrg'}, 'dct:license': 'https://creativecommons.org/licenses/by/4.0/', 'dct:created': '2025-01-23', 'vann:preferredNamespaceUri': 'https://example.com/supply-chain-document-workflow#', 'vann:preferredNamespacePrefix': 'scdw', 'owl:versionInfo': '1.0.0', 'dct:hasPart': [{'@id': 'https://example.com/supply-chain-document-workflow/core#', 'dct:title': 'Core Module', 'd

{'owl': 'http://www.w3.org/2002/07/owl#',
 'dct': 'http://purl.org/dc/terms/',
 'vann': 'http://purl.org/vocab/vann/',
 'foaf': 'http://xmlns.com/foaf/0.1/',
 'prov': 'http://www.w3.org/ns/prov#',
 'oplax': 'https://example.com/oplax#'}

## Role Prompting from Anthropic
[Giving Claude a role with a system prompt](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/system-prompts?utm_source=chatgpt.com#financial-analysis-with-role-prompting)

Prompt engineering

# Giving Claude a role with a system prompt

When using Claude, you can dramatically improve its performance by using the `system` parameter to give it a role. This technique, known as role prompting, is the most powerful way to use system prompts with Claude.

The right role can turn Claude from a general assistant into your virtual domain expert!

**System prompt tips** : Use the `system` parameter to set Claude’s role. Put everything else, like task-specific instructions, in the `user` turn instead.


Why use role prompting?

  * **Enhanced accuracy:** In complex scenarios like legal analysis or financial modeling, role prompting can significantly boost Claude’s performance.
  * **Tailored tone:** Whether you need a CFO’s brevity or a copywriter’s flair, role prompting adjusts Claude’s communication style.
  * **Improved focus:** By setting the role context, Claude stays more within the bounds of your task’s specific requirements.

How to give Claude a role

Use the `system` parameter in the Messages API to set Claude’s role:

```
import anthropic

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=2048,
    system="You are a seasoned data scientist at a Fortune 500 company.", # <-- role prompt
    messages=[
        {"role": "user", "content": "Analyze this dataset for anomalies: <dataset>{{DATASET}}</dataset>"}
    ]
)

print(response.content)

```



**Role prompting tip** : Experiment with roles! A `data scientist` might see different insights than a `marketing strategist` for the same data. A `data scientist specializing in customer insight analysis for Fortune 500 companies` might yield different results still!

## Self Correction Chains

[Chain complex prompts for stronger performance](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/chain-prompts#advanced-self-correction-chains)

# Chain complex prompts for stronger performance

When working with complex tasks, Claude can sometimes drop the ball if you try to handle everything in a single prompt. Chain of thought (CoT) prompting is great, but what if your task has multiple distinct steps that each require in-depth thought?

Enter prompt chaining: breaking down complex tasks into smaller, manageable subtasks.


Why chain prompts?

  1. **Accuracy** : Each subtask gets Claude’s full attention, reducing errors.
  2. **Clarity** : Simpler subtasks mean clearer instructions and outputs.
  3. **Traceability** : Easily pinpoint and fix issues in your prompt chain.

When to chain prompts

Use prompt chaining for multi-step tasks like research synthesis, document analysis, or iterative content creation. When a task involves multiple transformations, citations, or instructions, chaining prevents Claude from dropping or mishandling steps.

**Remember:** Each link in the chain gets Claude’s full attention!

**Debugging tip** : If Claude misses a step or performs poorly, isolate that step in its own prompt. This lets you fine-tune problematic steps without redoing the entire task
​
How to chain prompts

  1. **Identify subtasks** : Break your task into distinct, sequential steps.
  2. **Structure with XML for clear handoffs** : Use XML tags to pass outputs between prompts.
  3. **Have a single-task goal** : Each subtask should have a single, clear objective.
  4. **Iterate** : Refine subtasks based on Claude’s performance.
  Advanced: Self-correction chains

You can chain prompts to have Claude review its own work! This catches errors and refines outputs, especially for high-stakes tasks.

#### Prompt 1

Role| Content  
---|---  
User| Summarize this medical research paper.  
  
<paper>{{RESEARCH_PAPER}}</paper>  
  
Focus on methodology, findings, and clinical implications.  
Assistant  
{{SUMMARY}}| Methodology: Double-blind RCT with 500 Type 2 diabetes patients. Group A (n=250) received novel GLP-1/GIP dual agonist; Group B (n=250) received standard GLP-1 agonist. 24-week trial, primary endpoint: HbA1c reduction.  
  
Findings:  
\- Group A: 1.8% mean HbA1c reduction  
\- Group B: 1.1% mean HbA1c reduction  
\- 68% of Group A vs. 52% of Group B achieved HbA1c <7%  
\- Group A: 5.2kg mean weight loss  
\- Group B: 3.1kg mean weight loss  
  
Safety: Similar adverse event profiles, no serious events attributed to treatment.  
  
Clinical Implications: Dual agonist shows superior glycemic control and weight loss compared to standard therapy. Could be a game-changer, especially for obese diabetics.  
  
#### Prompt 2

Role| Content  
---|---  
User| Your task is to provide feedback on a research paper summary. Here is a summary of a medical research paper:  
<summary>  
{{SUMMARY}}  
</summary>  
  
Here is the research paper:  
<paper>  
{{RESEARCH_PAPER}}  
</paper>  
  
Review this summary for accuracy, clarity, and completeness on a graded A-F scale.  
Assistant  
{{FEEDBACK}}| 1\. Accuracy: A  
\- Correctly reported key stats: HbA1c reductions, weight loss.  
\- Accurately noted safety profile and lack of serious adverse events.  
\- Assertion of “game-changer” is justified by significantly better outcomes.  
  
2\. Clarity: A-  
\- Concise and well-structured with clear sections.  
\- Good use of comparative language (e.g., “up 3% from Q1”) for context.  
\- Minor issue: Acronyms like “RCT,” “GLP-1,” and “GIP” not defined. Consider spelling out first use.  
  
3\. Completeness: B+  
\- Covers all requested areas: methodology, findings, clinical implications.  
\- Missing: Baseline characteristics of groups (were they well-matched?).  
\- Missing: Any secondary endpoints (e.g., blood pressure, lipids).  
\- Consider: Long-term implications or need for further research.  
  
Overall, the summary is accurate and clear, providing key insights for a quick grasp of the study. To elevate to A+, define acronyms, mention group baseline stats, note any secondary outcomes, and briefly touch on the need for long-term studies.  
  
#### Prompt 3

Role| Content  
---|---  
User| Your task is to improve a paper summary given feedback. Here is the first draft of a medical research paper:  
<summary>  
{{SUMMARY}}  
</summary>  
  
Here is the research paper:  
<paper>  
{{RESEARCH_PAPER}}  
</paper>  
  
Here is the feedback:  
<feedback>  
{{FEEDBACK}}  
</feedback>  
  
Update the summary based on the feedback.  
Assistant| Methodology: Double-blind randomized controlled trial (RCT) with 500 Type 2 diabetes patients. Group A (n=250) received a novel glucagon-like peptide-1/glucose-dependent insulinotropic polypeptide (GLP-1/GIP) dual agonist; Group B (n=250) received a standard GLP-1 agonist. The 24-week trial’s primary endpoint was HbA1c reduction. Groups were well-matched for age, BMI, and baseline HbA1c.  
  
Findings:  
\- Group A: 1.8% mean HbA1c reduction  
\- Group B: 1.1% mean HbA1c reduction  
\- 68% of Group A vs. 52% of Group B achieved HbA1c <7%  
\- Group A: 5.2kg mean weight loss  
\- Group B: 3.1kg mean weight loss  
  
Secondary endpoints showed improvements in blood pressure and lipid profiles for both groups, with Group A showing slightly better results.  
  
Safety: Similar adverse event profiles, with no serious events attributed to either treatment.  
  
Clinical Implications: The dual GLP-1/GIP agonist demonstrates superior glycemic control and weight loss compared to standard GLP-1 therapy. These results suggest it could be a significant advancement, particularly for obese diabetics. However, longer-term studies are needed to confirm susta

**AI Prompt**

Ok. I think we want to create dynamic llm text båsed context for the llm to understand the JSON-LD structure so it can search and potentially reason over. We may want the LLM to review the context urls and do some introspection. The LLM should look at a prefix and uri and determine through prompting and LLM as Judge to gage if it is familiar. If not, it should attempt to retrieve the URI.

**AI Response**

I understand - let's create a tool-based workflow where the LLM can introspect JSON-LD contexts and make decisions about URI retrieval. We'll use Claudette's tools and role prompting to create this agentic system. Here's what I suggest:

```python
def analyze_prefix(
    prefix: str,  # Prefix to analyze (e.g. 'owl', 'dct')
    uri: str      # URI to analyze
) -> dict:        # Analysis of prefix familiarity
    "Analyzes a prefix and URI for familiarity"
    return {
        'prefix': prefix,
        'uri': uri,
        'is_w3c': 'w3.org' in uri,
        'is_purl': 'purl.org' in uri,
        'is_standard': any(s in uri for s in ['w3.org', 'purl.org', 'xmlns.com'])
    }

def inspect_context(
    ctx: dict  # Context dictionary to inspect
) -> list:    # List of prefix analyses
    "Inspects each prefix/URI pair in a context"
    return [analyze_prefix(p,u) for p,u in ctx.items()]
```

Now let's create a Chat instance with these tools and a specialized role:

```python
sp = """You are an expert knowledge graph engineer specializing in semantic web technologies. 
Your task is to analyze JSON-LD contexts and determine which URIs need further investigation.
For standard vocabularies (W3C, PURL, etc), you already know their structure.
For custom vocabularies, you should recommend retrieving and analyzing them."""

chat = Chat(model=models[1], sp=sp, tools=[analyze_prefix, inspect_context])
```

Let's try it with our sample context:

```python
chat.toolloop("""Here is a JSON-LD context to analyze:
<context>
{context}
</context>

Analyze each prefix/URI pair and recommend which ones need further investigation.""".format(context=context))
```

Would you like me to run this code and start building the introspection workflow? We can then enhance it with self-correction chains to improve the analysis quality.

In [ ]:
def analyze_prefix_uri(
    prefix: str,  # Prefix to analyze
    uri: str      # URI to analyze
) -> str:         # Analysis description
    "Analyzes a prefix and URI pair for familiarity"
    analysis = {
        'is_w3c': 'w3.org' in uri,
        'is_purl': 'purl.org' in uri,
        'is_standard': any(s in uri for s in ['w3.org', 'purl.org', 'xmlns.com'])
    }
    return f"Prefix '{prefix}' with URI {uri}: " + (
        "standard vocabulary" if analysis['is_standard'] 
        else "custom vocabulary, needs investigation"
    )

def analyze_vocabulary(
    vocab_uri: str  # URI of vocabulary to analyze
) -> str:          # Analysis of vocabulary
    "Attempts to retrieve and analyze a vocabulary URI"
    try:
        response = httpx.get(vocab_uri)
        return f"Retrieved vocabulary from {vocab_uri}. Status: {response.status_code}"
    except:
        return f"Could not retrieve vocabulary from {vocab_uri}"

sp = """You are an expert knowledge graph engineer specializing in semantic web technologies. 
Your task is to analyze JSON-LD contexts and determine which URIs need further investigation.
For standard vocabularies (W3C, PURL, etc), you already know their structure.
For custom vocabularies, you should recommend retrieving and analyzing them.
When you find a custom vocabulary, use the analyze_vocabulary tool to attempt retrieval."""

chat = Chat(model=models[1], sp=sp, tools=[analyze_prefix_uri, analyze_vocabulary])
prompt = """Analyze this prefix/URI pair:
prefix: oplax
uri: https://example.com/oplax#

Should we investigate this vocabulary further?"""

chat.toolloop(prompt)

Based on the results of our analysis:

1. This is a custom vocabulary using the example.com domain
2. The prefix "oplax" is not a commonly recognized prefix in the semantic web ecosystem
3. Attempting to retrieve the vocabulary returned a 404 error, indicating the vocabulary is not accessible at the given URI

Recommendations:
1. Since this is using example.com, this URI is likely just a placeholder and should be replaced with a proper, resolvable URI for production use
2. If this is meant to be a real vocabulary, you should:
   - Host the vocabulary at a permanent, accessible URI
   - Provide proper documentation for the terms defined in the vocabulary
   - Consider using a domain you control rather than example.com

Would you like to provide an alternative URI for this vocabulary that we could analyze?

<details>

- id: `msg_01R9hAen1DwDoq9ffebiQcQD`
- content: `[{'text': 'Based on the results of our analysis:\n\n1. This is a custom vocabulary using the example.com domain\n2. The prefix "oplax" is not a commonly recognized prefix in the semantic web ecosystem\n3. Attempting to retrieve the vocabulary returned a 404 error, indicating the vocabulary is not accessible at the given URI\n\nRecommendations:\n1. Since this is using example.com, this URI is likely just a placeholder and should be replaced with a proper, resolvable URI for production use\n2. If this is meant to be a real vocabulary, you should:\n   - Host the vocabulary at a permanent, accessible URI\n   - Provide proper documentation for the terms defined in the vocabulary\n   - Consider using a domain you control rather than example.com\n\nWould you like to provide an alternative URI for this vocabulary that we could analyze?', 'type': 'text'}]`
- model: `claude-3-5-sonnet-20241022`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 898, 'output_tokens': 182}`

</details>

The generalized semantic validation approach for JSON-LD has three core components:

Context Management:
- LLM receives JSON-LD contexts defining vocabularies and semantics
- System maintains active context definitions for validation
- Contexts establish property constraints, class hierarchies, and relationship rules

Semantic Validation:
- Types: Verify object types match context definitions
- Properties: Check property usage follows domain/range restrictions
- Relationships: Validate object connections respect semantic constraints
- Cardinality: Enforce relationship multiplicity rules
- Value Ranges: Confirm data values match type constraints

Repair Workflow:
1. LLM generates JSON-LD object based on request and contexts
2. Validator identifies semantic violations
3. LLM receives structured error feedback with context references
4. LLM repairs object while maintaining JSON-LD structure
5. Cycle continues until validation passes or max attempts reached

This approach enables semantic correctness while working with JSON-LD's flexible object representation format.

**AI Prompt**

Ok. I want to think through what is needed to provide context to LLMs to operate on a JSON-LD object. The JSON-LD may define a ontology which needs to be handled a bit differently than an instance graph that leverages an ontology. I want to think through the steps needed to provide context to LLMs to operate on a JSON-LD object.

**AI Response**

Let's break this down into a structured approach for handling both ontology definitions and instance data. Here's how I suggest we organize the context provision steps:

1. First-Level Analysis:
```python
def determine_jsonld_type(
    jsonld: dict  # Input JSON-LD document
) -> str:        # Returns 'ontology' or 'instance'
    "Determines if JSON-LD represents an ontology or instance data"
    graph = jsonld.get('@graph', [{}])[0]
    return 'ontology' if graph.get('@type') == 'owl:Ontology' else 'instance'
```

2. Ontology Context Handling:
```python
def analyze_ontology_structure(
    jsonld: dict  # Ontology JSON-LD
) -> dict:       # Ontology analysis results
    "Analyzes ontology structure for LLM context"
    return {
        'modules': [p for p in jsonld.get('@graph', [{}])[0].get('dct:hasPart', [])],
        'title': jsonld.get('@graph', [{}])[0].get('dct:title'),
        'description': jsonld.get('@graph', [{}])[0].get('dct:description'),
        'prompt_context': jsonld.get('@graph', [{}])[0].get('oplax:promptContext')
    }
```

Let's create a Chat instance with a specialized role for this task:

```python
ontology_sp = """You are an expert ontology engineer who understands both semantic web principles 
and how to explain ontological structures to other systems. Your task is to:
1. Determine if a JSON-LD document defines an ontology or contains instance data
2. For ontologies: analyze modular structure, identify key concepts, and extract LLM-friendly context
3. For instances: identify the ontologies referenced and extract relevant validation rules
4. Generate clear, structured explanations of the semantic context for other LLMs to use"""

chat = Chat(model=models[1], sp=ontology_sp, tools=[determine_jsonld_type, analyze_ontology_structure])
```

Would you like me to implement this initial structure and test it with your sample JSON-LD? We can then:
1. Add instance data handling
2. Implement context retrieval for referenced ontologies
3. Create a validation workflow
4. Add self-correction chains for improving context quality

What aspect would you like to explore first?

**AI Prompt**

Ok. We can also use types and isDescribedBy properties. It is obviously an ontology is it is rdf:type owl:Ontology. We probably want to create an in memory description for the LLM to understand the structure.

**AI Response**

Absolutely right! Let's enhance our analysis to include more RDF/OWL indicators and create a richer in-memory description for the LLM. Here's an improved approach:

```python
def analyze_type_indicators(
    jsonld: dict  # Input JSON-LD document
) -> dict:       # Type analysis results
    "Analyzes type indicators in JSON-LD"
    graph = jsonld.get('@graph', [{}])[0]
    return {
        'is_ontology': any(t in ['owl:Ontology', 'http://www.w3.org/2002/07/owl#Ontology'] 
                          for t in L(graph.get('@type', []))),
        'described_by': graph.get('rdfs:isDefinedBy', []) + graph.get('dcterms:isDescribedBy', []),
        'types': L(graph.get('@type', [])),
        'vocab': jsonld.get('@context', {}).get('@vocab')
    }

def build_llm_description(
    jsonld: dict  # Input JSON-LD document
) -> str:        # LLM-friendly description
    "Creates an LLM-friendly description of the structure"
    analysis = analyze_type_indicators(jsonld)
    structure = analyze_ontology_structure(jsonld)
    
    desc = ["This JSON-LD document represents:"]
    if analysis['is_ontology']:
        desc.extend([
            f"An ONTOLOGY titled '{structure['title']}'",
            f"Description: {structure['description']}",
            "\nModular Structure:",
            *[f"- {m['dct:title']}: {m['dct:description']}" for m in structure['modules']],
            "\nLLM Context:",
            structure['prompt_context']
        ])
    else:
        desc.extend([
            "INSTANCE DATA",
            f"Using vocabulary: {analysis['vocab']}" if analysis['vocab'] else "",
            "\nDescribed by:",
            *[f"- {uri}" for uri in analysis['described_by']],
            "\nTypes:",
            *[f"- {t}" for t in analysis['types']]
        ])
    
    return '\n'.join(desc)
```

Let's update our Chat instance to use this enhanced understanding:

```python
ontology_sp = """You are an expert ontology engineer who creates LLM-friendly semantic descriptions. Your task is to:
1. Analyze JSON-LD structural elements including type indicators, descriptions, and module relationships
2. Create clear, contextual descriptions that other LLMs can use for reasoning
3. Identify semantic patterns and constraints that should inform LLM operations
4. Maintain awareness of both ontology definitions and instance data patterns

For ontologies, focus on:
- Module relationships and dependencies
- Key concept hierarchies
- Property constraints and ranges
- Semantic validation rules

For instances, focus on:
- Type conformance
- Property usage patterns
- Relationship constraints
- Validation requirements"""

chat = Chat(model=models[1], sp=ontology_sp, 
           tools=[analyze_type_indicators, build_llm_description])
```

Would you like me to test this with your sample JSON-LD? We can see how the enhanced type detection and LLM-friendly description work, and then refine the approach based on the results.

**AI Prompt**

Ok. This is open world, so not every ontology will follow the FAIR guidelines for Ontologies and Vocabularies. In addition, there are different levels of formality with the most widely used profile being similar to schema.org in addition to RDFS, RDFS+, OWLRL, etc. We should be able to handle this.

**AI Response**

Excellent point! Let's create a more flexible system that can handle different levels of ontology formality and completeness. Here's an approach that considers these variations:

```python
def detect_profile(
    jsonld: dict  # Input JSON-LD document
) -> dict:       # Profile analysis
    "Detects the semantic expressivity profile"
    graph = jsonld.get('@graph', [{}])[0]
    constructs = set()
    
    # Look for indicators of different profiles
    if any('owl:' in str(k) for k in graph.keys()): constructs.add('owl')
    if any('rdfs:' in str(k) for k in graph.keys()): constructs.add('rdfs')
    if any('schema:' in str(k) for k in graph.keys()): constructs.add('schema')
    
    # Detect specific OWL features
    owl_features = {
        'restrictions': any('owl:Restriction' in str(v) for v in graph.values()),
        'class_axioms': any(k in str(graph) for k in ['owl:disjointWith', 'owl:equivalentClass']),
        'property_chains': 'owl:propertyChainAxiom' in str(graph)
    }
    
    return {
        'profile': 'owl-dl' if owl_features['restrictions'] or owl_features['class_axioms']
                  else 'owl-rl' if constructs & {'owl'} 
                  else 'rdfs+' if constructs & {'rdfs'} 
                  else 'schema-like' if constructs & {'schema'}
                  else 'minimal',
        'constructs': list(constructs),
        'owl_features': owl_features if 'owl' in constructs else None
    }

def analyze_fair_compliance(
    jsonld: dict  # Input JSON-LD document
) -> dict:       # FAIR compliance indicators
    "Analyzes FAIR principles compliance"
    graph = jsonld.get('@graph', [{}])[0]
    
    return {
        'findable': {
            'has_identifier': bool(graph.get('@id')),
            'has_metadata': any(k in graph for k in ['dct:title', 'rdfs:label', 'schema:name']),
            'has_searchable_metadata': any(k in graph for k in ['dct:subject', 'dct:description'])
        },
        'accessible': {
            'has_protocol': 'http' in str(graph.get('@id', '')),
            'has_license': any(k in graph for k in ['dct:license', 'cc:license'])
        },
        'interoperable': {
            'uses_standards': bool(jsonld.get('@context')),
            'has_vocab_links': any(k in graph for k in ['rdfs:isDefinedBy', 'owl:imports'])
        },
        'reusable': {
            'has_provenance': any(k in graph for k in ['dct:creator', 'prov:wasGeneratedBy']),
            'has_usage_info': any(k in graph for k in ['vann:usageNote', 'schema:usageInfo'])
        }
    }

def build_flexible_description(
    jsonld: dict  # Input JSON-LD document
) -> str:        # Flexible LLM-friendly description
    "Creates description accounting for varying formality levels"
    profile = detect_profile(jsonld)
    fair = analyze_fair_compliance(jsonld)
    
    desc = [f"This is a {profile['profile']} level semantic description"]
    
    # Add available metadata based on what's present
    if fair['findable']['has_metadata']:
        desc.append("\nCore Information:")
        for k in ['dct:title', 'rdfs:label', 'schema:name']:
            if v := jsonld.get('@graph', [{}])[0].get(k):
                desc.append(f"- {k.split(':')[1]}: {v}")
    
    # Note semantic features
    if profile['constructs']:
        desc.append("\nSemantic Features:")
        desc.extend(f"- Uses {c} vocabulary" for c in profile['constructs'])
        
    if profile['owl_features']:
        desc.append("\nOWL Features Present:")
        desc.extend(f"- {k.replace('_',' ')}" 
                   for k,v in profile['owl_features'].items() if v)
    
    # Add FAIR assessment
    desc.append("\nFAIR Assessment:")
    for principle, checks in fair.items():
        score = sum(checks.values()) / len(checks)
        desc.append(f"- {principle.title()}: {score*100:.0f}% compliance")
    
    return '\n'.join(desc)
```

Let's update our chat system to handle this flexibility:

```python
flexible_sp = """You are an expert semantic web engineer who understands different levels of ontology formality. Your task is to:
1. Analyze semantic descriptions ranging from minimal vocabularies to full OWL-DL ontologies
2. Adapt your reasoning based on the formality level detected
3. Extract maximum value from available metadata while gracefully handling missing elements
4. Provide guidance on potential semantic enhancements when appropriate

Remember:
- Schema.org-style vocabularies focus on practical usage over formal axioms
- RDFS adds basic class hierarchies and domain/range constraints
- OWL adds complex class expressions and logical constraints
- Not all vocabularies will follow FAIR principles completely
- Focus on available information while noting potential improvements"""

chat = Chat(model=models[1], sp=flexible_sp, 
           tools=[detect_profile, analyze_fair_compliance, build_flexible_description])
```

Would you like me to test this with some different types of vocabularies to see how it handles varying levels of formality? We could try it with:
1. A schema.org-style vocabulary
2. An RDFS vocabulary
3. A full OWL ontology
4. A minimal vocabulary with limited metadata

In [ ]:
def detect_profile(
    jsonld: dict  # Input JSON-LD document
) -> dict:       # Profile analysis
    "Detects the semantic expressivity profile"
    graph = jsonld.get('@graph', [{}])[0]
    constructs = set()
    
    # Look for indicators of different profiles
    if any('owl:' in str(k) for k in graph.keys()): constructs.add('owl')
    if any('rdfs:' in str(k) for k in graph.keys()): constructs.add('rdfs')
    if any('schema:' in str(k) for k in graph.keys()): constructs.add('schema')
    
    # Detect specific OWL features
    owl_features = {
        'restrictions': any('owl:Restriction' in str(v) for v in graph.values()),
        'class_axioms': any(k in str(graph) for k in ['owl:disjointWith', 'owl:equivalentClass']),
        'property_chains': 'owl:propertyChainAxiom' in str(graph)
    }
    
    return {
        'profile': 'owl-dl' if owl_features['restrictions'] or owl_features['class_axioms']
                  else 'owl-rl' if constructs & {'owl'} 
                  else 'rdfs+' if constructs & {'rdfs'} 
                  else 'schema-like' if constructs & {'schema'}
                  else 'minimal',
        'constructs': list(constructs),
        'owl_features': owl_features if 'owl' in constructs else None
    }

def analyze_fair_compliance(
    jsonld: dict  # Input JSON-LD document
) -> dict:       # FAIR compliance indicators
    "Analyzes FAIR principles compliance"
    graph = jsonld.get('@graph', [{}])[0]
    
    return {
        'findable': {
            'has_identifier': bool(graph.get('@id')),
            'has_metadata': any(k in graph for k in ['dct:title', 'rdfs:label', 'schema:name']),
            'has_searchable_metadata': any(k in graph for k in ['dct:subject', 'dct:description'])
        },
        'accessible': {
            'has_protocol': 'http' in str(graph.get('@id', '')),
            'has_license': any(k in graph for k in ['dct:license', 'cc:license'])
        },
        'interoperable': {
            'uses_standards': bool(jsonld.get('@context')),
            'has_vocab_links': any(k in graph for k in ['rdfs:isDefinedBy', 'owl:imports'])
        },
        'reusable': {
            'has_provenance': any(k in graph for k in ['dct:creator', 'prov:wasGeneratedBy']),
            'has_usage_info': any(k in graph for k in ['vann:usageNote', 'schema:usageInfo'])
        }
    }

def build_flexible_description(
    jsonld: dict  # Input JSON-LD document
) -> str:        # Flexible LLM-friendly description
    "Creates description accounting for varying formality levels"
    profile = detect_profile(jsonld)
    fair = analyze_fair_compliance(jsonld)
    
    desc = [f"This is a {profile['profile']} level semantic description"]
    
    # Add available metadata based on what's present
    if fair['findable']['has_metadata']:
        desc.append("\nCore Information:")
        for k in ['dct:title', 'rdfs:label', 'schema:name']:
            if v := jsonld.get('@graph', [{}])[0].get(k):
                desc.append(f"- {k.split(':')[1]}: {v}")
    
    # Note semantic features
    if profile['constructs']:
        desc.append("\nSemantic Features:")
        desc.extend(f"- Uses {c} vocabulary" for c in profile['constructs'])
        
    if profile['owl_features']:
        desc.append("\nOWL Features Present:")
        desc.extend(f"- {k.replace('_',' ')}" 
                   for k,v in profile['owl_features'].items() if v)
    
    # Add FAIR assessment
    desc.append("\nFAIR Assessment:")
    for principle, checks in fair.items():
        score = sum(checks.values()) / len(checks)
        desc.append(f"- {principle.title()}: {score*100:.0f}% compliance")
    
    return '\n'.join(desc)
flexible_sp = """You are an expert semantic web engineer who understands different levels of ontology formality. Your task is to:
1. Analyze semantic descriptions ranging from minimal vocabularies to full OWL-DL ontologies
2. Adapt your reasoning based on the formality level detected
3. Extract maximum value from available metadata while gracefully handling missing elements
4. Provide guidance on potential semantic enhancements when appropriate

Remember:
- Schema.org-style vocabularies focus on practical usage over formal axioms
- RDFS adds basic class hierarchies and domain/range constraints
- OWL adds complex class expressions and logical constraints
- Not all vocabularies will follow FAIR principles completely
- Focus on available information while noting potential improvements"""

chat = Chat(model=models[1], sp=flexible_sp, 
           tools=[detect_profile, analyze_fair_compliance, build_flexible_description])

In [ ]:
flexible_sp = """You are an expert semantic web engineer who understands different levels of ontology formality. Your task is to:
1. Analyze semantic descriptions ranging from minimal vocabularies to full OWL-DL ontologies
2. Adapt your reasoning based on the formality level detected
3. Extract maximum value from available metadata while gracefully handling missing elements
4. Provide guidance on potential semantic enhancements when appropriate

Remember:
- Schema.org-style vocabularies focus on practical usage over formal axioms
- RDFS adds basic class hierarchies and domain/range constraints
- OWL adds complex class expressions and logical constraints
- Not all vocabularies will follow FAIR principles completely
- Focus on available information while noting potential improvements"""

chat = Chat(model=models[1], sp=flexible_sp, 
           tools=[detect_profile, analyze_fair_compliance, build_flexible_description])

**AI Prompt**

What am I missing, this doesn't seem to have done anything?

**AI Response**

Ah, we've defined the functions and set up the Chat instance, but we haven't actually run any analysis yet! Let's test it with your sample JSON-LD that we fetched earlier:

```python
# Let's analyze the supply chain ontology we fetched earlier
result = chat.toolloop("""Please analyze this JSON-LD document:
<jsonld>
{jsonld}
</jsonld>

Provide a detailed analysis of its structure, formality level, and FAIR compliance.""".format(jsonld=jsonld))

print(result)
```

This should run our analysis pipeline:
1. `detect_profile` will identify the formality level (OWL, RDFS, etc.)
2. `analyze_fair_compliance` will check FAIR principles
3. `build_flexible_description` will create an LLM-friendly description

Then the LLM will provide its interpretation and recommendations based on the analysis.

Would you like me to run this? We can then see what aspects of the analysis might need improvement, such as:
- Additional OWL feature detection
- More detailed module relationship analysis
- Better handling of vocabulary imports
- Enhanced metadata extraction

**AI Prompt**

This prompt didn't seem to work

**AI Response**

Ah, I see the issue - we've hit a problem with the toolloop trying to process the dictionary directly. Let's modify our approach slightly:

```python
# First, let's test our individual functions
doc_type = detect_profile(jsonld)
print("\nProfile Detection:")
print(doc_type)

fair_analysis = analyze_fair_compliance(jsonld)
print("\nFAIR Analysis:")
print(fair_analysis)

description = build_flexible_description(jsonld)
print("\nFull Description:")
print(description)
```

This will help us:
1. See if our individual functions work correctly
2. Identify where any issues might be
3. Understand what the analysis is telling us about the JSON-LD

Then we can fix the chat interface if needed. Would you like me to run this diagnostic code?

The error we got suggests there's an issue with how we're passing the JSON-LD to the toolloop function. We might need to modify our approach to handle the JSON-LD structure more carefully, perhaps by serializing it differently or adjusting how we pass it to the tools.

In [ ]:
# First, let's test our individual functions
doc_type = detect_profile(jsonld)
print("\nProfile Detection:")
print(doc_type)

fair_analysis = analyze_fair_compliance(jsonld)
print("\nFAIR Analysis:")
print(fair_analysis)

description = build_flexible_description(jsonld)
print("\nFull Description:")
print(description)


Profile Detection:
{'profile': 'owl-rl', 'constructs': ['owl'], 'owl_features': {'restrictions': False, 'class_axioms': False, 'property_chains': False}}

FAIR Analysis:
{'findable': {'has_identifier': True, 'has_metadata': True, 'has_searchable_metadata': True}, 'accessible': {'has_protocol': True, 'has_license': True}, 'interoperable': {'uses_standards': True, 'has_vocab_links': False}, 'reusable': {'has_provenance': False, 'has_usage_info': False}}

Full Description:
This is a owl-rl level semantic description

Core Information:
- title: Supply Chain Document Workflow Ontology

Semantic Features:
- Uses owl vocabulary

OWL Features Present:

FAIR Assessment:
- Findable: 100% compliance
- Accessible: 100% compliance
- Interoperable: 50% compliance
- Reusable: 0% compliance


**AI Prompt**

That seemed to work. Now what?

**AI Response**

Great! Now we can see our analysis is working. Let's enhance it to be more useful for LLMs. I notice a few areas we can improve:

1. Add module analysis - we're not fully capturing the modular structure we see in the sample
2. Add better OWL detection - we're seeing it's OWL-RL but missing some nuances
3. Create a more LLM-friendly context summary

Here's an improved version:

```python
def analyze_modules(
    jsonld: dict  # Input JSON-LD document
) -> dict:       # Module analysis
    "Analyzes modular structure of ontology"
    graph = jsonld.get('@graph', [{}])[0]
    modules = graph.get('dct:hasPart', [])
    return {
        'module_count': len(modules),
        'modules': [{
            'id': m.get('@id'),
            'title': m.get('dct:title'),
            'description': m.get('dct:description')
        } for m in modules],
        'prompt_context': graph.get('oplax:promptContext')
    }

def build_llm_context(
    jsonld: dict  # Input JSON-LD document
) -> str:        # LLM-friendly context string
    "Builds comprehensive context for LLM understanding"
    profile = detect_profile(jsonld)
    fair = analyze_fair_compliance(jsonld)
    modules = analyze_modules(jsonld)
    
    context = [
        f"This ontology ({jsonld['@graph'][0].get('dct:title', 'Untitled')}) is:",
        f"1. A {profile['profile']} level semantic model",
        f"2. Composed of {modules['module_count']} modules:",
        *[f"   - {m['title']}: {m['description']}" for m in modules['modules']],
        "\nKey characteristics:",
        f"- Uses {', '.join(profile['constructs'])} constructs",
        f"- FAIR scores: Findable {sum(fair['findable'].values())/3*100:.0f}%, " +
        f"Accessible {sum(fair['accessible'].values())/2*100:.0f}%, " +
        f"Interoperable {sum(fair['interoperable'].values())/2*100:.0f}%, " +
        f"Reusable {sum(fair['reusable'].values())/2*100:.0f}%",
        "\nLLM Usage Context:",
        modules['prompt_context']
    ]
    
    return '\n'.join(context)
```

Let's update our chat system to use this enhanced analysis:

```python
enhanced_sp = """You are an expert semantic web engineer who helps other LLMs understand ontology structures.
Your task is to analyze semantic models and provide clear guidance on:
1. How to interpret the ontology's structure and modules
2. What semantic features are available for reasoning
3. What constraints and patterns should be followed
4. How to validate instance data against this model

Base your analysis on the provided context, noting both strengths and potential areas for enhancement."""

chat = Chat(model=models[1], sp=enhanced_sp, 
           tools=[analyze_modules, build_llm_context])

# Let's try with our sample
print(build_llm_context(jsonld))
```

Would you like me to run this enhanced version? We can then use it as a foundation for:
1. Better instance data validation
2. Module relationship analysis
3. Pattern detection
4. Constraint checking

In [ ]:
def analyze_modules(
    jsonld: dict  # Input JSON-LD document
) -> dict:       # Module analysis
    "Analyzes modular structure of ontology"
    graph = jsonld.get('@graph', [{}])[0]
    modules = graph.get('dct:hasPart', [])
    return {
        'module_count': len(modules),
        'modules': [{
            'id': m.get('@id'),
            'title': m.get('dct:title'),
            'description': m.get('dct:description')
        } for m in modules],
        'prompt_context': graph.get('oplax:promptContext')
    }

def build_llm_context(
    jsonld: dict  # Input JSON-LD document
) -> str:        # LLM-friendly context string
    "Builds comprehensive context for LLM understanding"
    profile = detect_profile(jsonld)
    fair = analyze_fair_compliance(jsonld)
    modules = analyze_modules(jsonld)
    
    context = [
        f"This ontology ({jsonld['@graph'][0].get('dct:title', 'Untitled')}) is:",
        f"1. A {profile['profile']} level semantic model",
        f"2. Composed of {modules['module_count']} modules:",
        *[f"   - {m['title']}: {m['description']}" for m in modules['modules']],
        "\nKey characteristics:",
        f"- Uses {', '.join(profile['constructs'])} constructs",
        f"- FAIR scores: Findable {sum(fair['findable'].values())/3*100:.0f}%, " +
        f"Accessible {sum(fair['accessible'].values())/2*100:.0f}%, " +
        f"Interoperable {sum(fair['interoperable'].values())/2*100:.0f}%, " +
        f"Reusable {sum(fair['reusable'].values())/2*100:.0f}%",
        "\nLLM Usage Context:",
        modules['prompt_context']
    ]
    
    return '\n'.join(context)
enhanced_sp = """You are an expert semantic web engineer who helps other LLMs understand ontology structures.
Your task is to analyze semantic models and provide clear guidance on:
1. How to interpret the ontology's structure and modules
2. What semantic features are available for reasoning
3. What constraints and patterns should be followed
4. How to validate instance data against this model

Base your analysis on the provided context, noting both strengths and potential areas for enhancement."""

chat = Chat(model=models[1], sp=enhanced_sp, 
           tools=[analyze_modules, build_llm_context])

# Let's try with our sample
print(build_llm_context(jsonld))

This ontology (Supply Chain Document Workflow Ontology) is:
1. A owl-rl level semantic model
2. Composed of 4 modules:
   - Core Module: Foundation classes and properties for the supply chain document workflow.
   - Ordering Module: Defines concepts and relations specific to purchase orders.
   - Billing Module: Defines concepts and relations specific to invoices and payment details.
   - Logistics Module: Defines concepts and relations specific to shipping, delivery, and logistics.

Key characteristics:
- Uses owl constructs
- FAIR scores: Findable 100%, Accessible 100%, Interoperable 50%, Reusable 0%

LLM Usage Context:
This ontology is the main entry point for the Supply Chain Document Workflow. Refer to the modules for specific classes and properties. LLMs can use this information to guide queries, generate documentation, or perform transformations involving supply chain documents.


**AI Prompt**

So, I think we may also want to classify potential affordances that are defined within the JSON-LD object. For instance, schema.org defines things like "Action" and "Thing" as affordances. I think we should be able to detect these affordances and then use them to generate a list of affordances that are available in the JSON-LD object. This will allow us to generate a list of affordances that are available in the JSON-LD. Also, there are things like url, and dataset download.

**AI Response**

Excellent point! Let's create an affordance detection system that can identify both semantic affordances (like Actions, Things) and practical affordances (like downloadable resources, API endpoints). Here's an approach:

```python
def detect_affordances(
    jsonld: dict  # Input JSON-LD document
) -> dict:       # Detected affordances
    "Detects both semantic and practical affordances in JSON-LD"
    graph = jsonld.get('@graph', [{}])[0]
    
    # Common semantic type patterns
    semantic_patterns = {
        'actions': ['Action', 'schema:Action', 'void:Dataset'],
        'things': ['Thing', 'schema:Thing', 'owl:Thing'],
        'agents': ['Agent', 'foaf:Agent', 'prov:Agent'],
        'events': ['Event', 'schema:Event', 'prov:Activity'],
        'services': ['Service', 'schema:Service', 'hydra:ApiDocumentation']
    }
    
    # Practical affordance patterns
    practical_patterns = {
        'downloads': ['void:dataDump', 'dcat:downloadURL', 'schema:downloadUrl'],
        'endpoints': ['void:sparqlEndpoint', 'hydra:entrypoint', 'schema:url'],
        'documentation': ['schema:documentation', 'rdfs:seeAlso', 'vann:usageNote'],
        'examples': ['vann:example', 'skos:example', 'schema:example'],
        'apis': ['hydra:apiDocumentation', 'schema:webAPI', 'void:openSearchDescription']
    }
    
    def find_patterns(patterns, obj):
        found = {}
        for k, terms in patterns.items():
            matches = []
            for term in terms:
                # Check direct properties
                if any(term in str(p) for p in obj.keys()):
                    matches.extend(v for k,v in obj.items() if term in str(k))
                # Check type declarations
                if '@type' in obj and any(term in str(t) for t in L(obj['@type'])):
                    matches.append(obj.get('@id', 'unnamed'))
            if matches: found[k] = matches
        return found
    
    return {
        'semantic': find_patterns(semantic_patterns, graph),
        'practical': find_patterns(practical_patterns, graph),
        'vocabulary_terms': {
            'classes': [v for k,v in graph.items() if 'rdfs:Class' in str(v) or 'owl:Class' in str(v)],
            'properties': [v for k,v in graph.items() if 'rdf:Property' in str(v)],
            'instances': [v for k,v in graph.items() if '@type' in str(v) and not any(x in str(v) for x in ['Class', 'Property'])]
        }
    }

def build_affordance_context(
    jsonld: dict  # Input JSON-LD document
) -> str:        # LLM-friendly affordance description
    "Creates LLM-friendly description of available affordances"
    affordances = detect_affordances(jsonld)
    
    lines = ["Available Affordances:"]
    
    if affordances['semantic']:
        lines.append("\nSemantic Affordances:")
        for k,v in affordances['semantic'].items():
            lines.append(f"- {k.title()}: {len(v)} found")
            lines.extend(f"  * {item}" for item in v[:3])
            if len(v) > 3: lines.append(f"  * ... and {len(v)-3} more")
    
    if affordances['practical']:
        lines.append("\nPractical Affordances:")
        for k,v in affordances['practical'].items():
            lines.append(f"- {k.title()}: {len(v)} found")
            lines.extend(f"  * {item}" for item in v[:3])
            if len(v) > 3: lines.append(f"  * ... and {len(v)-3} more")
    
    vocab = affordances['vocabulary_terms']
    lines.append("\nVocabulary Structure:")
    lines.extend([
        f"- Classes: {len(vocab['classes'])}",
        f"- Properties: {len(vocab['properties'])}",
        f"- Instances: {len(vocab['instances'])}"
    ])
    
    return '\n'.join(lines)
```

Let's update our main context builder to include affordances:

```python
def build_enhanced_llm_context(
    jsonld: dict  # Input JSON-LD document
) -> str:        # Enhanced LLM-friendly context
    "Builds comprehensive context including affordances"
    return build_llm_context(jsonld) + "\n\n" + build_affordance_context(jsonld)

# Let's try it with our sample
print(build_enhanced_llm_context(jsonld))
```

This enhancement:
1. Detects semantic affordances (Actions, Things, etc.)
2. Identifies practical affordances (downloads, endpoints, etc.)
3. Analyzes vocabulary structure
4. Presents everything in an LLM-friendly format

Would you like me to run this and see what affordances it detects in your sample? We can then refine the patterns based on what we find.

In [ ]:
def detect_affordances(
    jsonld: dict  # Input JSON-LD document
) -> dict:       # Detected affordances
    "Detects both semantic and practical affordances in JSON-LD"
    graph = jsonld.get('@graph', [{}])[0]
    
    # Common semantic type patterns
    semantic_patterns = {
        'actions': ['Action', 'schema:Action', 'void:Dataset'],
        'things': ['Thing', 'schema:Thing', 'owl:Thing'],
        'agents': ['Agent', 'foaf:Agent', 'prov:Agent'],
        'events': ['Event', 'schema:Event', 'prov:Activity'],
        'services': ['Service', 'schema:Service', 'hydra:ApiDocumentation']
    }
    
    # Practical affordance patterns
    practical_patterns = {
        'downloads': ['void:dataDump', 'dcat:downloadURL', 'schema:downloadUrl'],
        'endpoints': ['void:sparqlEndpoint', 'hydra:entrypoint', 'schema:url'],
        'documentation': ['schema:documentation', 'rdfs:seeAlso', 'vann:usageNote'],
        'examples': ['vann:example', 'skos:example', 'schema:example'],
        'apis': ['hydra:apiDocumentation', 'schema:webAPI', 'void:openSearchDescription']
    }
    
    def find_patterns(patterns, obj):
        found = {}
        for k, terms in patterns.items():
            matches = []
            for term in terms:
                # Check direct properties
                if any(term in str(p) for p in obj.keys()):
                    matches.extend(v for k,v in obj.items() if term in str(k))
                # Check type declarations
                if '@type' in obj and any(term in str(t) for t in L(obj['@type'])):
                    matches.append(obj.get('@id', 'unnamed'))
            if matches: found[k] = matches
        return found
    
    return {
        'semantic': find_patterns(semantic_patterns, graph),
        'practical': find_patterns(practical_patterns, graph),
        'vocabulary_terms': {
            'classes': [v for k,v in graph.items() if 'rdfs:Class' in str(v) or 'owl:Class' in str(v)],
            'properties': [v for k,v in graph.items() if 'rdf:Property' in str(v)],
            'instances': [v for k,v in graph.items() if '@type' in str(v) and not any(x in str(v) for x in ['Class', 'Property'])]
        }
    }

def build_affordance_context(
    jsonld: dict  # Input JSON-LD document
) -> str:        # LLM-friendly affordance description
    "Creates LLM-friendly description of available affordances"
    affordances = detect_affordances(jsonld)
    
    lines = ["Available Affordances:"]
    
    if affordances['semantic']:
        lines.append("\nSemantic Affordances:")
        for k,v in affordances['semantic'].items():
            lines.append(f"- {k.title()}: {len(v)} found")
            lines.extend(f"  * {item}" for item in v[:3])
            if len(v) > 3: lines.append(f"  * ... and {len(v)-3} more")
    
    if affordances['practical']:
        lines.append("\nPractical Affordances:")
        for k,v in affordances['practical'].items():
            lines.append(f"- {k.title()}: {len(v)} found")
            lines.extend(f"  * {item}" for item in v[:3])
            if len(v) > 3: lines.append(f"  * ... and {len(v)-3} more")
    
    vocab = affordances['vocabulary_terms']
    lines.append("\nVocabulary Structure:")
    lines.extend([
        f"- Classes: {len(vocab['classes'])}",
        f"- Properties: {len(vocab['properties'])}",
        f"- Instances: {len(vocab['instances'])}"
    ])
    
    return '\n'.join(lines)
def build_enhanced_llm_context(
    jsonld: dict  # Input JSON-LD document
) -> str:        # Enhanced LLM-friendly context
    "Builds comprehensive context including affordances"
    return build_llm_context(jsonld) + "\n\n" + build_affordance_context(jsonld)

# Let's try it with our sample
print(build_enhanced_llm_context(jsonld))

This ontology (Supply Chain Document Workflow Ontology) is:
1. A owl-rl level semantic model
2. Composed of 4 modules:
   - Core Module: Foundation classes and properties for the supply chain document workflow.
   - Ordering Module: Defines concepts and relations specific to purchase orders.
   - Billing Module: Defines concepts and relations specific to invoices and payment details.
   - Logistics Module: Defines concepts and relations specific to shipping, delivery, and logistics.

Key characteristics:
- Uses owl constructs
- FAIR scores: Findable 100%, Accessible 100%, Interoperable 50%, Reusable 0%

LLM Usage Context:
This ontology is the main entry point for the Supply Chain Document Workflow. Refer to the modules for specific classes and properties. LLMs can use this information to guide queries, generate documentation, or perform transformations involving supply chain documents.

Available Affordances:

Vocabulary Structure:
- Classes: 0
- Properties: 0
- Instances: 0


In [ ]:
def build_enhanced_llm_context(
    jsonld: dict  # Input JSON-LD document
) -> str:        # Enhanced LLM-friendly context
    "Builds comprehensive context including affordances"
    return build_llm_context(jsonld) + "\n\n" + build_affordance_context(jsonld)

# Let's try it with our sample
print(build_enhanced_llm_context(jsonld))

This ontology (Supply Chain Document Workflow Ontology) is:
1. A owl-rl level semantic model
2. Composed of 4 modules:
   - Core Module: Foundation classes and properties for the supply chain document workflow.
   - Ordering Module: Defines concepts and relations specific to purchase orders.
   - Billing Module: Defines concepts and relations specific to invoices and payment details.
   - Logistics Module: Defines concepts and relations specific to shipping, delivery, and logistics.

Key characteristics:
- Uses owl constructs
- FAIR scores: Findable 100%, Accessible 100%, Interoperable 50%, Reusable 0%

LLM Usage Context:
This ontology is the main entry point for the Supply Chain Document Workflow. Refer to the modules for specific classes and properties. LLMs can use this information to guide queries, generate documentation, or perform transformations involving supply chain documents.

Available Affordances:

Vocabulary Structure:
- Classes: 0
- Properties: 0
- Instances: 0


**AI Prompt**

From this exploration, I'd like to start to construct a basic sort of methodology and procedure for LLMS to structure their own understanding of a JSON-LD object. I'd like to start with a basic outline of the steps that an LLM should take to understand a JSON-LD object and the various catagories of contextual interpretation. Let's not write any code, but rather reflect on this exploration.

**AI Response**

Based on our exploration, here's a proposed methodology for LLMs to systematically understand JSON-LD objects:

1. Document Type Classification
   - Determine if it's an ontology or instance data
   - Identify the formality level (Schema-like, RDFS, OWL-RL, OWL-DL)
   - Check if it's a standalone document or part of a modular structure

2. Structural Analysis
   - Parse the @context declarations to understand namespace bindings
   - Identify modular components and their relationships
   - Recognize the organizational patterns (hierarchical, networked, etc.)
   - Map dependencies between components

3. Semantic Capability Assessment
   - Identify available semantic constructs (classes, properties, axioms)
   - Detect semantic affordances (Actions, Things, Events)
   - Understand reasoning capabilities based on expressivity level
   - Map constraints and validation rules

4. Practical Affordance Inventory
   - Locate documentation resources
   - Identify API endpoints and services
   - Find example data and usage patterns
   - Map downloadable resources and external references

5. Contextual Integration
   - Understand how this JSON-LD relates to known vocabularies
   - Identify reuse of standard patterns
   - Recognize domain-specific adaptations
   - Map to familiar concepts and analogies

6. Usage Pattern Analysis
   - Identify intended usage scenarios
   - Understand validation requirements
   - Map query and interaction patterns
   - Recognize common operations and transformations

7. Quality Assessment
   - Evaluate FAIR principle compliance
   - Check for completeness of documentation
   - Assess semantic consistency
   - Identify potential gaps or ambiguities

This methodology helps LLMs build a comprehensive mental model of the JSON-LD object, enabling them to:
- Reason about its structure and capabilities
- Make informed decisions about how to use it
- Identify potential issues or limitations
- Suggest improvements or extensions
- Guide users in proper usage

Would you like to explore any of these aspects in more detail or discuss how they might be formalized into more specific guidelines?

**AI Prompt**

We used tool use primarily for this exploration exercise, but this may not necessarily be the most efficient way to store the information for a large language model context. Let's reflect on that and what are our options for structuring the underlying information so that it's interpretable to a large language model.

**AI Response**

Excellent point! From our exploration, I see several potential approaches for structuring this information for LLM consumption, each with different trade-offs:

1. Natural Language Descriptions
- Pros:
  * Directly consumable by LLMs without translation
  * Can include nuanced explanations and relationships
  * Easier to maintain and update
- Cons:
  * Less structured than formal representations
  * May introduce ambiguity
  * Harder to validate programmatically

2. Structured Templates
```
ONTOLOGY: {name}
TYPE: {formality_level}
MODULES:
  - {module_name}: {purpose}
AFFORDANCES:
  Semantic: {list}
  Practical: {list}
```
- Pros:
  * Consistent format for LLM parsing
  * Balance between structure and readability
  * Easy to update systematically
- Cons:
  * May miss nuanced relationships
  * Less flexible than natural language

3. Knowledge Graph Format
```
{ontology} hasType {formality_level}
{ontology} hasModule {module}
{module} provides {affordance}
```
- Pros:
  * Explicit relationship representation
  * Matches semantic web principles
  * Good for reasoning tasks
- Cons:
  * May be overly formal for simple uses
  * Requires more processing by LLM

4. Hybrid Contextual Format
```
Overview: {natural language description}
Structure:
  {structured template}
Relationships:
  {graph-like representations}
Usage Guidelines:
  {natural language examples}
```
- Pros:
  * Combines benefits of multiple approaches
  * Provides multiple ways for LLM to understand
  * More robust for different types of queries
- Cons:
  * More complex to maintain
  * Potentially redundant information

For LLM consumption, I believe the Hybrid Contextual Format might be most effective because:
1. It provides multiple "views" of the same information
2. LLMs can choose the most appropriate representation for their task
3. It maintains both structure and flexibility
4. It supports both quick reference and deep understanding

Would you like to explore how we might implement one of these approaches, or discuss other potential structuring strategies?

**AI Prompt**

Okay, let's summarize what we've done here in this SolveIt session.

**AI Response**

In this SolveIt session, we explored how to help LLMs understand and work with JSON-LD documents. Here's what we accomplished:

1. Initial Exploration
- Created tools to detect and analyze JSON-LD document types
- Built functions to assess FAIR principles compliance
- Developed methods to analyze ontology modularity

2. Enhanced Analysis
- Added affordance detection for both semantic and practical capabilities
- Created flexible profile detection for different levels of formality
- Built LLM-friendly context descriptions

3. Methodology Development
- Created a systematic approach for LLMs to understand JSON-LD objects
- Developed a 7-step analysis framework covering:
  * Document classification
  * Structural analysis
  * Semantic capabilities
  * Practical affordances
  * Contextual integration
  * Usage patterns
  * Quality assessment

4. Context Structure Analysis
- Explored different approaches for presenting information to LLMs:
  * Natural language descriptions
  * Structured templates
  * Knowledge graph format
  * Hybrid contextual format
- Evaluated pros and cons of each approach

5. Key Insights
- JSON-LD understanding needs multiple layers of analysis
- Different formality levels require flexible approaches
- LLMs benefit from having multiple representations of the same information
- The importance of balancing structure with accessibility

This exploration provides a foundation for:
- Building more sophisticated JSON-LD analysis tools
- Creating better LLM-friendly documentation
- Developing standardized approaches for semantic web interaction with LLMs
- Improving machine-readable metadata practices